# RG3 패턴별RF들을실제1개다중클래스분류기로통합
RF/ET/LR 실제최대3개. 전처리회귀와학습군집은분류기수와구분해서보고한다.


In [1]:
from pathlib import Path
import nbformat,itertools,json
import numpy as np,pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
n=nbformat.read("notebooks/187_rg3_gas_max3_classifier_refit.ipynb",as_version=4)
small_source="\n".join(c.source for c in n.cells if c.cell_type=="code")
exec(small_source[:small_source.index('names=["RF_std4"')])
names=["RF_mode8","ET_thermal_mode6","LR_pressure_mode6"]
variants=[v for k in [1,2,3] for v in itertools.combinations(names,k)]
# Reuse training-normal conditional feature construction, not187 fitted classifiers.
exec(small_source[small_source.index("localmemo={}"):small_source.index("memo={};audits=[]")])
memo={};audits=[]
def fit_head(x,yy,q,name):
 imp=SimpleImputer(keep_empty_features=True).fit(x)
 a=imp.transform(x);b=imp.transform(q)
 sc=RobustScaler().fit(a[yy==0])
 z=sc.transform(a)
 positive=np.flatnonzero(yy==1)
 k=min(2,len(positive),len(np.unique(z[positive],axis=0)))
 mode=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(z[positive]) if k>1 else np.zeros(len(positive),int)
 labels=np.zeros(len(yy),int);labels[positive]=mode+1
 # One multiclass classifier learns all positive modes. KMeans supplies TRAIN labels only.
 if name=="RF_mode8":m=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)
 elif name=="ET_thermal_mode6":m=ExtraTreesClassifier(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)
 else:m=LogisticRegression(C=.1,class_weight="balanced",max_iter=10000,random_state=42);a=np.clip(z,-20,20);b=np.clip(sc.transform(b),-20,20)
 m.fit(a,labels)
 return m.predict_proba(b)[:,m.classes_>0].sum(axis=1)
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 tr_res=np.full((len(ref),4),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  tr_res[mask]=res_features(ref[~mask],ref[mask])
 te_res=res_features(ref,q)
 train8=np.column_stack([d.iloc[ref][stdcols],tr_res])
 query8=np.column_stack([d.iloc[q][stdcols],te_res])
 out={"RF_mode8":fit_head(train8,y[ref],query8,"RF_mode8")};zo={}
 for kind,name in [("thermal","ET_thermal_mode6"),("pressure","LR_pressure_mode6")]:
  tr=local(ref,ref,kind);te=local(ref,q,kind);zo[kind]=te
  out[name]=fit_head(tr,y[ref],te,name)
 audits.append({"train":len(ref),"train_gas":int(y[ref].sum()),"query":len(q),
 "timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "shared_comparison_block":len(set(groups.iloc[ref])&set(groups.iloc[q]))})
 memo[key]=(out,zo);return memo[key]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(2*tp/(2*tp+fp+fn)) if tp else 0,
 "precision":float(tp/(tp+fp)) if tp+fp else 0,"recall":float(tp/(tp+fn)) if tp+fn else 0}
records=[];choices=[];components=[]
for held in blocks:
 q=np.flatnonzero(groups.eq(held));ref=np.flatnonzero(~groups.eq(held))
 outer,zo=predict(ref,q);inner={n:np.full(len(ref),np.nan) for n in names}
 zi={k:np.full((len(ref),6),np.nan) for k in ["thermal","pressure"]}
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  ip,z=predict(ref[~mask],ref[mask])
  for n in names:inner[n][mask]=ip[n]
  for k in zi:zi[k][mask]=z[k]
 ip={};op={}
 for n in names:
  quant=.975 if n=="ET_thermal_mode6" else .99
  cut=float(np.quantile(inner[n][y[ref]==0],quant,method="higher"))
  ip[n]=inner[n]>cut;op[n]=outer[n]>cut
  if n=="ET_thermal_mode6":
   gate=float(np.quantile(zi["thermal"][y[ref]==0,4],.05))
   ip[n]&=zi["thermal"][:,4]<gate;op[n]&=zo["thermal"][:,4]<gate
  if n=="LR_pressure_mode6":
   def direction(ids):
    a=d.iloc[ids]
    return ((a.Max_Injection_Pressure__delta1.round(2)<0)&(a.Max_Injection_Speed__delta1.round(2)>0)&(a.Max_Switch_Over_Pressure__delta1.round(2)<a.Max_Injection_Pressure__delta1.round(2))).to_numpy()
   ip[n]&=direction(ref);op[n]&=direction(q)
  for j,i in enumerate(q):components.append({"head":n,"TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"score":float(outer[n][j]),"threshold":cut,"prediction":int(op[n][j])})
 opts=[]
 for combo in variants:
  label="+".join(combo)
  pred=np.any(np.column_stack([op[n] for n in combo]),axis=1)
  met=stats(y[ref],np.any(np.column_stack([ip[n] for n in combo]),axis=1))
  if met["FP"]<=int(np.floor((y[ref]==0).sum()*.03)):
   opts.append((met["F1"],met["TP"],-met["FP"],-len(combo),label))
  for j,i in enumerate(q):records.append({"policy":label,"model_count":len(combo),"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"prediction":int(pred[j])})
 best=max(opts) if opts else (0,0,0,0,"none")
 chosen=best[-1];combo=chosen.split("+")
 pred=np.zeros(len(q),bool) if chosen=="none" else np.any(np.column_stack([op[n] for n in combo]),axis=1)
 choices.append({"heldout_block":held,"selected":chosen,"count":0 if chosen=="none" else len(combo),"inner_F1":best[0]})
 for j,i in enumerate(q):records.append({"policy":"primary_inner_selected_max3","model_count":0 if chosen=="none" else len(combo),"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"prediction":int(pred[j])})
 print("Mode-consolidated small classifiers",held,chosen,flush=True)
pp=pd.DataFrame(records);rows=[]
for policy,a in pp.groupby("policy",sort=False):
 assert len(a)==583 and a.TimeStamp.is_unique
 rows.append({"policy":policy,"classifiers_max":int(a.model_count.max()),"N":583,"gas":17,**stats(a.actual,a.prediction)})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"rg3_188_consolidated_mode_predictions.csv",index=False)
mm.to_csv(OUT/"rg3_188_consolidated_mode_metrics.csv",index=False)
pd.DataFrame(components).to_csv(OUT/"rg3_188_consolidated_single_head_predictions.csv",index=False)
pd.DataFrame(audits).to_csv(OUT/"rg3_188_consolidated_fit_audit.csv",index=False)
pd.DataFrame(choices).to_csv(OUT/"rg3_188_consolidated_inner_choices.csv",index=False)
assert pd.DataFrame(audits).timestamp_overlap.eq(0).all() and pd.DataFrame(audits).shared_comparison_block.eq(0).all()
print("\nMETRICS\n",mm.to_string(index=False))
top=mm[mm.policy!="primary_inner_selected_max3"].sort_values(["F1","FP","classifiers_max"],ascending=[False,True,True])
best=top.iloc[0]
cases=pp[pp.policy.eq(best.policy)].merge(d[["TimeStamp","Reason","comparison_block"]],on="TimeStamp",validate="one_to_one")
cases[cases.actual.eq(1)].to_csv(OUT/"rg3_188_best_fixed_all17_gas.csv",index=False)
print("\nALL GAS\n",cases[cases.actual.eq(1)][["TimeStamp","prediction"]].to_string(index=False))
lines=["# RG3 축소2차: 패턴별모델을다중클래스1개로통합","",
 "정상/가스패턴A/가스패턴B를한분류기에서학습한다. 가스패턴은해당학습가스에서만KMeans2로정한다.평가가스를군집학습에넣지않는다.",
 "실제분류기는RF8/ET열6/LR압력6의최대3개다.시드반복·동일알고리즘여러개를평균하지않는다.",
 "RF8=기존흔들림4＋정상관계잔차4, ET열6/LR압력6=기존물리피처. 잔차는정상회귀4개로계산하는전처리이므로분류기수와별도로밝힌다.",
 "KMeans는학습가스의보조라벨을만드는데만쓰고추론때별도탐지기로쓰지않는다. 정상이력Ridge4회귀/최근접검색등전처리를분류기3개와혼동하지않는다.",
 "기존583샷/5구간, 안쪽검증으로기준과최대3개조합선택. 사후고정상위와주모델을구분한다.",
 "이미본평가자료에서의개발결과이며새Run독립성능은아니다.","",
 "## 전체조합",mm.to_string(index=False),"",
 "## 고정개발최고의가스17개",cases[cases.actual.eq(1)][["TimeStamp","prediction"]].to_string(index=False)]
(ROOT/"reports/rg3_188_consolidated_modes_comparison.md").write_text("\n".join(lines),encoding="utf-8")


Mode-consolidated small classifiers 20201021_001626_RG3 LR_pressure_mode6


Mode-consolidated small classifiers 20201022_002405_RG3 LR_pressure_mode6


Mode-consolidated small classifiers 20201022_044026_RG3 LR_pressure_mode6


Mode-consolidated small classifiers 20201023_010128_RG3 RF_mode8+ET_thermal_mode6


Mode-consolidated small classifiers 20201023_044148_RG3 RF_mode8



METRICS
                                      policy  classifiers_max   N  gas  TP  FP  FN  TN       F1  precision   recall
                                   RF_mode8                1 583   17   0   4  17 562 0.000000   0.000000 0.000000
                           ET_thermal_mode6                1 583   17   1   0  16 566 0.111111   1.000000 0.058824
                          LR_pressure_mode6                1 583   17   0   1  17 565 0.000000   0.000000 0.000000
                  RF_mode8+ET_thermal_mode6                2 583   17   1   4  16 562 0.090909   0.200000 0.058824
                 RF_mode8+LR_pressure_mode6                2 583   17   0   5  17 561 0.000000   0.000000 0.000000
         ET_thermal_mode6+LR_pressure_mode6                2 583   17   1   1  16 565 0.105263   0.500000 0.058824
RF_mode8+ET_thermal_mode6+LR_pressure_mode6                3 583   17   1   5  16 561 0.086957   0.166667 0.058824
                primary_inner_selected_max3                2 583   17 

2015